<a href="https://colab.research.google.com/github/Saiji/Data-Science-Work/blob/master/Kaggle_Churn_Prediction_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# ── STEP 1: Load data ──────────────────────────────────────────────────────
train = pd.read_csv('/content/train.csv')
test  = pd.read_csv('/content/test.csv')

# ── STEP 2: Encode features ────────────────────────────────────────────────
def preprocess(df):
    df = df.copy()

    # Yes/No columns → 0 or 1
    yn_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    if 'Churn' in df.columns:
        yn_cols.append('Churn')
    for col in yn_cols:
        df[col] = (df[col] == 'Yes').astype(int)

    # gender → 0 or 1
    df['gender'] = (df['gender'] == 'Male').astype(int)

    # Service columns: "No internet service" / "No phone service" → 0, No → 1, Yes → 2
    service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    for col in service_cols:
        df[col] = df[col].map({'Yes': 2, 'No': 1,
                               'No phone service': 0, 'No internet service': 0})

    # Ordinal encoding for multi-class categoricals
    df['InternetService'] = df['InternetService'].map({'Fiber optic': 2, 'DSL': 1, 'No': 0})
    df['Contract']        = df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
    df['PaymentMethod']   = df['PaymentMethod'].map({
        'Electronic check': 0, 'Mailed check': 1,
        'Bank transfer (automatic)': 2, 'Credit card (automatic)': 3
    })

    # ── STEP 3: Engineered features ────────────────────────────────────────
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)  # avoid /0
    df['ChargeRatio']      = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['TenureGroup']      = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                                    labels=[0,1,2,3]).astype(float)
    df['NumServices']      = df[service_cols].apply(lambda r: (r == 2).sum(), axis=1)
    df['HasFiber']         = (df['InternetService'] == 2).astype(int)
    df['IsMonthToMonth']   = (df['Contract'] == 0).astype(int)

    return df

train_p = preprocess(train)
test_p  = preprocess(test)

feature_cols = [c for c in train_p.columns if c not in ['id', 'Churn']]
X      = train_p[feature_cols].values   # shape: (594194, 22)
y      = train_p['Churn'].values        # 0 or 1
X_test = test_p[feature_cols].values

# ── STEP 4: Train the model ────────────────────────────────────────────────
model = lgb.LGBMClassifier(
    n_estimators=300,       # 300 decision trees
    learning_rate=0.1,      # each tree contributes 10% of its score
    num_leaves=63,          # max leaves per tree (controls complexity)
    subsample=0.8,          # use 80% of rows per tree (reduces overfitting)
    colsample_bytree=0.8,   # use 80% of columns per tree
    min_child_samples=50,
    random_state=42,
    n_jobs=4,
    verbose=-1
)
model.fit(X, y)

# ── STEP 5: Predict probabilities ─────────────────────────────────────────
# predict_proba returns shape (n_rows, 2):
#   column 0 = P(Churn = No)
#   column 1 = P(Churn = Yes)  ← this is what we want
proba_matrix = model.predict_proba(X_test)
# e.g. [[0.9, 0.1],   ← row 0: 90% no-churn, 10% churn
#        [0.7, 0.3],   ← row 1: 70% no-churn, 30% churn
#        [0.8, 0.2]]   ← row 2: 80% no-churn, 20% churn

churn_probs = proba_matrix[:, 1]   # pick the "Yes" column → [0.1, 0.3, 0.2, ...]

# ── STEP 6: How does LightGBM compute 0.1 internally? ─────────────────────
# Each of the 300 trees votes a small log-odds score.
# The scores are summed → raw_score (e.g. -2.197)
# Then sigmoid converts it to a probability:
#
#   P = 1 / (1 + e^(-raw_score))
#   P = 1 / (1 + e^(2.197))  ≈  0.1   (10% churn chance)
#
# You can verify:
raw_score = np.log(0.1 / (1 - 0.1))   # = -2.197
recovered_prob = 1 / (1 + np.exp(-raw_score))
print(f"raw score: {raw_score:.3f}  →  sigmoid → prob: {recovered_prob:.3f}")
# raw score: -2.197  →  sigmoid → prob: 0.100

# ── STEP 7: Build and save submission ─────────────────────────────────────
submission = pd.DataFrame({
    'id':    test_p['id'],
    'Churn': churn_probs          # values like 0.1, 0.3, 0.2 ...
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
# id       Churn
# 594194   0.056
# 594195   0.001
# 594196   0.111

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


raw score: -2.197  →  sigmoid → prob: 0.100
       id     Churn
0  594194  0.025558
1  594195  0.000045
2  594196  0.124897
3  594197  0.001419
4  594198  0.442323
